# ICA Algorithm: Learning the Unmixing Matrix

## Learning Objectives
- Choose a **source density** $p_s(s) = g'(s)$ where $g$ is a sigmoid — captures non-Gaussianity
- Derive the **log-likelihood** $\ell(W) = \sum_i\left[\sum_j \log g'(w_j^T x^{(i)}) + \log|W|\right]$
- Derive the **stochastic gradient update** for $W$
- Understand why the update has two terms: a **score function** term and a **log-determinant** term
- Apply ICA to the cocktail party problem and verify source recovery

## Problem Statement

**Source density.** Each source $s_j$ has density modelled as:
$$p_s(s) = g'(s), \qquad g(s) = \frac{1}{1+e^{-s}} \text{ (sigmoid)}$$

This is the logistic/sigmoid density — non-Gaussian, with heavier tails than Gaussian.

**Log-likelihood.** From the density transformation $p_x(x) = \prod_j p_s(w_j^T x) \cdot |W|$:
$$\ell(W) = \sum_{i=1}^m \left[\sum_{j=1}^n \log g'(w_j^T x^{(i)}) + \log|\det W|\right]$$

**Stochastic gradient ascent update** (one example $x^{(i)}$ at a time):
$$W := W + \alpha \left(\begin{bmatrix} 1 - 2g(w_1^T x^{(i)}) \\ \vdots \\ 1 - 2g(w_n^T x^{(i)}) \end{bmatrix} {x^{(i)}}^T + (W^T)^{-1}\right)$$

where:
- $[1 - 2g(w_j^T x^{(i)})]\, {x^{(i)}}^T$ — gradient of $\sum_j \log g'(w_j^T x^{(i)})$ w.r.t. $W$
- $(W^T)^{-1}$ — gradient of $\log|\det W|$ w.r.t. $W$

**Key identity.** $\frac{\partial}{\partial W}\log|\det W| = (W^{-1})^T = (W^T)^{-1}$

## 1. The Sigmoid Source Density

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, laplace

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_density(x):
    # g'(s) = g(s)(1-g(s)) — derivative of sigmoid
    g = sigmoid(x)
    return g * (1.0 - g)

x = np.linspace(-6, 6, 500)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: sigmoid and its derivative
ax = axes[0]
ax.plot(x, sigmoid(x),         '#2166ac', lw=2.5, label='$g(s) = 1/(1+e^{-s})$ (sigmoid)')
ax.plot(x, sigmoid_density(x), '#d6604d', lw=2.5, label="$g'(s) = g(s)(1-g(s))$ (density)")
ax.axhline(0.5, color='gray', ls=':', lw=1)
ax.set_xlabel('$s$')
ax.set_title('Sigmoid and its derivative\n$g(s)$ is CDF, $g\'(s)$ is PDF')
ax.legend(fontsize=9)

# Middle: compare source densities
ax = axes[1]
ax.plot(x, sigmoid_density(x),       '#d6604d', lw=2.5, label='Sigmoid density $g\'(s)$')
ax.plot(x, norm.pdf(x),              '#2166ac', lw=2.5, label='Gaussian $\\mathcal{N}(0,1)$')
ax.plot(x, laplace.pdf(x, scale=1/np.sqrt(2)), '#4dac26', lw=2.5, label='Laplace (super-Gaussian)')
ax.set_xlabel('$s$')
ax.set_ylabel('Density')
ax.set_title('Source density comparison\nSigmoid: heavier tails than Gaussian')
ax.legend(fontsize=9)

# Right: score function 1 - 2g(s)
ax = axes[2]
score = 1 - 2 * sigmoid(x)
ax.plot(x, score, '#4dac26', lw=2.5)
ax.axhline(0, color='k', ls='--', lw=1)
ax.axvline(0, color='gray', ls=':', lw=1)
ax.fill_between(x, score, 0, where=score > 0, alpha=0.2, color='green', label='Push $w_j$ up')
ax.fill_between(x, score, 0, where=score < 0, alpha=0.2, color='red',   label='Push $w_j$ down')
ax.set_xlabel('$w_j^T x^{(i)}$')
ax.set_ylabel('$1 - 2g(w_j^T x^{(i)})$')
ax.set_title('Score function in gradient update\n$1-2g(s)$: sign of gradient signal')
ax.legend(fontsize=9)

fig.suptitle('ICA: Sigmoid Source Density and Score Function',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. ICA Algorithm: Stochastic Gradient Ascent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def ica_sgd(X, alpha=0.005, n_epochs=10, seed=None):
    """
    ICA via stochastic gradient ascent on log-likelihood.
    Source density: p_s(s) = g'(s) = sigmoid(s)*(1-sigmoid(s)).
    Update: W += alpha * ([1-2g(Wx)] x^T + (W^T)^{-1})
    """
    rng = np.random.default_rng(seed)
    m, n = X.shape
    W = np.eye(n) + rng.standard_normal((n, n)) * 0.01
    ll_hist = []

    for epoch in range(n_epochs):
        idx = rng.permutation(m)
        for i in idx:
            xi = X[i]                           # (n,)
            s  = W @ xi                          # (n,)
            g  = sigmoid(s)                      # (n,)
            # Score term
            score = (1.0 - 2.0 * g)[:, None] * xi[None, :]  # (n,n)
            # Log-det term
            try:
                log_det_grad = np.linalg.inv(W).T
            except np.linalg.LinAlgError:
                continue
            W = W + alpha * (score + log_det_grad)

        # Log-likelihood estimate on a subset
        subset = X[:200]
        S_sub  = subset @ W.T
        log_g_prime = np.log(sigmoid(S_sub) * (1 - sigmoid(S_sub)) + 1e-300)
        sign, logdet = np.linalg.slogdet(W)
        ll = log_g_prime.sum() + len(subset) * logdet
        ll_hist.append(ll)

    return W, ll_hist

# Generate cocktail party data
np.random.seed(42)
m = 2000
t = np.linspace(0, 10, m)
s1 = np.sin(2 * np.pi * 1.1 * t)
s2 = (t % 1.0) * 2 - 1             # sawtooth in [-1, 1]
S  = np.column_stack([s1, s2])

A = np.array([[0.8, 0.5], [0.3, 0.9]])
X = S @ A.T

# Whiten X (standard preprocessing for ICA)
Xc = X - X.mean(axis=0)
U, sv, Vt = np.linalg.svd(Xc, full_matrices=False)
X_white = U * np.sqrt(m)   # whitened: E[xx^T] = I

W_hat, ll_hist = ica_sgd(X_white, alpha=0.002, n_epochs=20, seed=0)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Log-likelihood
ax = axes[0, 0]
ax.plot(range(1, len(ll_hist)+1), ll_hist, 'b-o', lw=2.5, ms=5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Log-likelihood (subset)')
ax.set_title('ICA log-likelihood during training')

# Scatter: observed vs recovered
S_rec = X_white @ W_hat.T
# Align signs
for j in range(2):
    if np.corrcoef(S_rec[:, j], S[:, j])[0, 1] < 0:
        S_rec[:, j] = -S_rec[:, j]

ax = axes[0, 1]
ax.scatter(S_rec[:, 0], S_rec[:, 1], s=3, alpha=0.3, c='steelblue')
ax.set_xlabel('Recovered $s_1$'); ax.set_ylabel('Recovered $s_2$')
ax.set_title('Recovered source space\n(should look rectangular like original sources)')

# Time series: original vs recovered
for j, (row, col, sig, title) in enumerate([
    (1, 0, S_rec[:, 0], 'Recovered source 1'),
    (1, 1, S_rec[:, 1], 'Recovered source 2'),
]):
    ax = axes[row, col]
    ax.plot(t[:300], S[:300, j], 'k-', lw=1, alpha=0.5, label='True source')
    ax.plot(t[:300], sig[:300],  'r--', lw=1.5, label='ICA recovered')
    corr = abs(np.corrcoef(sig, S[:, j])[0, 1])
    ax.set_title(f'{title}\ncorr with true = {corr:.4f}')
    ax.legend(fontsize=8.5)
    ax.set_xlabel('Time')

fig.suptitle('ICA via Stochastic Gradient Ascent: Cocktail Party Recovery',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

corr1 = abs(np.corrcoef(S_rec[:, 0], S[:, 0])[0, 1])
corr2 = abs(np.corrcoef(S_rec[:, 1], S[:, 1])[0, 1])
print(f'Source 1 recovery correlation: {corr1:.4f}')
print(f'Source 2 recovery correlation: {corr2:.4f}')

## 3. Gradient Derivation: Score Term and Log-Determinant Term

The gradient of the log-likelihood with respect to $W$ has two parts:

$$\nabla_W \ell = \underbrace{\sum_i \begin{bmatrix}(1-2g(w_1^Tx^{(i)}))x^{(i)T}\\ \vdots \\ (1-2g(w_n^Tx^{(i)}))x^{(i)T}\end{bmatrix}}_{\text{score term: from }\log g'(w_j^Tx)} + \underbrace{m(W^T)^{-1}}_{\text{log-det term: from }\log|W|}$$

**Key identity:** $\frac{\partial}{\partial W}\log|\det W| = (W^{-1})^T = (W^T)^{-1}$

This can be verified as $\det W \cdot (W^{-1})^T$ from the cofactor expansion / Jacobi's formula.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

np.random.seed(1)

# Verify gradient numerically
def ica_ll(W, X):
    """Log-likelihood for ICA with sigmoid source density."""
    m, n = X.shape
    S  = X @ W.T
    g  = sigmoid(S)
    ll = np.log(g * (1 - g) + 1e-300).sum()
    sign, logdet = np.linalg.slogdet(W)
    ll += m * logdet
    return ll

def ica_grad_analytic(W, X):
    """Analytic gradient of ICA log-likelihood."""
    m, n = X.shape
    S  = X @ W.T              # (m, n)
    g  = sigmoid(S)            # (m, n)
    score_term = ((1 - 2*g).T @ X)  # (n, n)
    logdet_term = m * np.linalg.inv(W).T
    return score_term + logdet_term

def ica_grad_numeric(W, X, eps=1e-5):
    """Numeric gradient via finite differences."""
    grad = np.zeros_like(W)
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            W_plus = W.copy(); W_plus[i, j] += eps
            W_minus = W.copy(); W_minus[i, j] -= eps
            grad[i, j] = (ica_ll(W_plus, X) - ica_ll(W_minus, X)) / (2 * eps)
    return grad

m_test = 100
n_test = 3
X_test = np.random.randn(m_test, n_test)
W_test = np.eye(n_test) + np.random.randn(n_test, n_test) * 0.3

g_analytic = ica_grad_analytic(W_test, X_test)
g_numeric  = ica_grad_numeric(W_test, X_test)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Compare gradient terms
ax = axes[0]
S_test  = X_test @ W_test.T
g       = sigmoid(S_test)
score   = ((1 - 2*g).T @ X_test)
logdet  = m_test * np.linalg.inv(W_test).T
terms   = [score.flatten(), logdet.flatten(), g_analytic.flatten()]
labels  = ['Score term', 'Log-det term', 'Total gradient']
ax.boxplot(terms, tick_labels=labels)
ax.axhline(0, color='k', ls='--', lw=1)
ax.set_title('Gradient components\n(score + log-det = total)')
ax.set_ylabel('Gradient value')

# Analytic vs numeric gradient
ax = axes[1]
ax.scatter(g_numeric.flatten(), g_analytic.flatten(), s=40, alpha=0.8, c='steelblue')
mn, mx = g_numeric.min(), g_numeric.max()
ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='y=x (perfect)')
ax.set_xlabel('Numeric gradient')
ax.set_ylabel('Analytic gradient')
ax.set_title('Gradient check: analytic vs numeric')
ax.legend(fontsize=9)
err = np.max(np.abs(g_analytic - g_numeric))
ax.text(0.05, 0.95, f'Max error: {err:.2e}', transform=ax.transAxes,
        va='top', fontsize=10, bbox=dict(boxstyle='round', fc='lightyellow'))

# Log-det gradient identity: d/dW log|det W| = (W^T)^{-1}
ax = axes[2]
W_rand = np.random.randn(3, 3) + 2*np.eye(3)  # well-conditioned
# Numeric
eps = 1e-5
grad_logdet_num = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        W_p = W_rand.copy(); W_p[i,j] += eps
        W_m = W_rand.copy(); W_m[i,j] -= eps
        grad_logdet_num[i,j] = (np.log(abs(np.linalg.det(W_p))) -
                                 np.log(abs(np.linalg.det(W_m)))) / (2*eps)
# Analytic: (W^T)^{-1}
grad_logdet_ana = np.linalg.inv(W_rand).T

ax.scatter(grad_logdet_num.flatten(), grad_logdet_ana.flatten(), s=60, c='#d6604d')
mn, mx = grad_logdet_num.min(), grad_logdet_num.max()
ax.plot([mn, mx], [mn, mx], 'k--', lw=2)
ax.set_xlabel('Numeric $\\partial \\log|\\det W| / \\partial W_{ij}$')
ax.set_ylabel('Analytic $(W^T)^{-1}_{ij}$')
ax.set_title('Identity: $\\nabla_W \\log|\\det W| = (W^T)^{-1}$')
err2 = np.max(np.abs(grad_logdet_ana - grad_logdet_num))
ax.text(0.05, 0.95, f'Max error: {err2:.2e}', transform=ax.transAxes,
        va='top', fontsize=10, bbox=dict(boxstyle='round', fc='lightyellow'))

fig.suptitle('ICA Gradient: Score Term + Log-Determinant Term',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. ICA vs PCA: Rotation Recovery

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import FastICA, PCA

np.random.seed(42)
m = 3000
t = np.linspace(0, 20, m)

# Three independent non-Gaussian sources
s1 = np.sin(2 * np.pi * 0.7 * t)         # sine
s2 = np.sign(np.sin(2 * np.pi * 1.3 * t)) # square wave
s3 = (t % 1.5) / 1.5 * 2 - 1             # sawtooth
S  = np.column_stack([s1, s2, s3])

# Mix
A = np.array([[0.8, 0.4, 0.3],
              [0.2, 0.9, 0.1],
              [0.5, 0.1, 0.8]])
X = S @ A.T

# ICA and PCA
ica  = FastICA(n_components=3, random_state=0, max_iter=1000)
pca  = PCA(n_components=3)
S_ica = ica.fit_transform(X)
S_pca = pca.fit_transform(X)

# Align sign for correlation
def align_sign(S_rec, S_true):
    S_out = S_rec.copy()
    for j in range(S_true.shape[1]):
        corr = np.corrcoef(S_rec[:, j], S_true[:, j])[0, 1]
        if corr < 0:
            S_out[:, j] = -S_rec[:, j]
    return S_out

S_ica_aligned = align_sign(S_ica, S)
S_pca_aligned = align_sign(S_pca, S)

fig, axes = plt.subplots(4, 3, figsize=(15, 12))

show = 400
titles_row = ['True sources', 'Mixed observations', 'ICA recovered', 'PCA recovered']
colors = ['#2166ac', '#d6604d', '#4dac26']

for col in range(3):
    # True sources
    axes[0, col].plot(t[:show], S[:show, col], color=colors[col], lw=1.5)
    axes[0, col].set_title(f'Source {col+1}' if col == 0 else f'Source {col+1}', fontsize=9)
    axes[0, col].set_ylabel(titles_row[0] if col == 0 else '')

    # Mixed
    axes[1, col].plot(t[:show], X[:show, col], color='gray', lw=1.5)
    axes[1, col].set_ylabel(titles_row[1] if col == 0 else '')

    # ICA
    corr_ica = np.max([abs(np.corrcoef(S_ica[:, col], S[:, j])[0, 1]) for j in range(3)])
    axes[2, col].plot(t[:show], S_ica_aligned[:show, col], color='#d6604d', lw=1.5)
    axes[2, col].set_title(f'corr={corr_ica:.3f}', fontsize=8)
    axes[2, col].set_ylabel(titles_row[2] if col == 0 else '')

    # PCA
    corr_pca = np.max([abs(np.corrcoef(S_pca[:, col], S[:, j])[0, 1]) for j in range(3)])
    axes[3, col].plot(t[:show], S_pca_aligned[:show, col], color='steelblue', lw=1.5)
    axes[3, col].set_title(f'corr={corr_pca:.3f}', fontsize=8)
    axes[3, col].set_ylabel(titles_row[3] if col == 0 else '')
    axes[3, col].set_xlabel('Time')

fig.suptitle('ICA vs PCA: 3-Source Cocktail Party\n'
             'ICA recovers independent sources; PCA only decorrelates',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Recovery correlations (best-match per ICA component):')
for j in range(3):
    c_ica = max(abs(np.corrcoef(S_ica[:, j], S[:, k])[0,1]) for k in range(3))
    c_pca = max(abs(np.corrcoef(S_pca[:, j], S[:, k])[0,1]) for k in range(3))
    print(f'  Component {j+1}: ICA={c_ica:.4f}, PCA={c_pca:.4f}')

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Source density</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">p(s) = g&#x2032;(s)</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >choose sigmoid g(s) = 1/(1+e&#x207b;ˢ) &#x2192; p(s) = g&#x2032;(s) (derivative)</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >write log-likelihood in terms of W</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Log-likelihood</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">&#x2113;(W)</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >&#x2113;(W) = &#x2211;&#x2071; [&#x2211;&#x2c7; log g&#x2032;(w&#x2c7;&#x1d40;x&#x2071;) + log|det W|]</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >take gradient with respect to W</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Gradient</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">d&#x2113;/dW</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >&#x2207;&#x2096;&#x2113; = (1-2g(Wx))x&#x1d40; + (W&#x1d40;)&#x207b;&#xb9; &#x2014; score + log-det terms</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >stochastic gradient ascent</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Converged W</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >W = A&#x207b;&#xb9;: sources s = Wx are independent &#x2014; ICA unmixes the signals</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| Source density | $p(s) = g'(s) = g(s)(1-g(s))$ where $g$ is sigmoid | Non-Gaussian; unimodal with heavy tails |
| Log-likelihood | $\ell(W) = \sum_i \left[\sum_j \log g'(w_j^T x^{(i)}) + \log\lvert\det W\rvert\right]$ | Two terms: source independence + Jacobian |
| Gradient | $\nabla_W \ell = (1 - 2g(Wx))x^T + (W^T)^{-1}$ | Score function + log-determinant gradient |
| Update rule | $W := W + \alpha \nabla_W \ell$ (stochastic, one $x^{(i)}$ at a time) | Stochastic gradient ascent |
| Result | $W = A^{-1}$, recovered sources $s = Wx$ | Columns of $A$ recovered up to scale/permutation |

**Key insight:** ICA maximises the log-likelihood by gradient ascent; the gradient has a score term (pushes sources toward the assumed distribution) and a log-det term (prevents the trivial $W=0$ solution) — together they drive the unmixing matrix toward $A^{-1}$.